In [1]:
import openai
import pandas as pd
import numpy as np
import re
import sys, os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import ast
import re
import json
from tqdm import tqdm

In [2]:
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[0]
sys.path.append(str(repo_path))

In [3]:
from py.utils import verifyDir

In [4]:
API_KEY = os.environ.get('OPENAI_API_KEY', '')
DATA_PATH = f"{repo_path}/data/"
ARTICLES_PATH = f"{DATA_PATH}articles/"
CSV_PATH=f"{repo_path}/outputs/csv/"

In [5]:
verifyDir(CSV_PATH)

### Loading graph

In [6]:
df = pd.read_csv(f"{DATA_PATH}csv/graph.csv", sep=",", low_memory=False)
df["source"] = df["source"].apply(lambda x: x.lower())
df["type"] = df["type"].apply(lambda x: x.lower())
df["target"] = df["target"].apply(lambda x: x.lower())
df

,source,source_type,type,weight,target,target_type,_last_edited_by,_last_edited_date,_date_added,_date,_raw_source,_algorithm,_articleid
0,frey inc,Entity.Organization.FishingCompany,event.invest,1,sustainable_nets,Entity.Commodity,Jack Inch,2035-02-01 01:00:00,2035-02-01 01:00:00,2035-02-01,The News Buoy,BassLine,Frey Inc__0__0__The News Buoy
1,city of port grove,Entity.Organization.GovernmentOrg,event.certificateissued,1,barnes and sons,Entity.Organization.FishingCompany,Kristin Baker,2035-02-01 01:00:00,2035-02-01 01:00:00,2035-02-01,Lomark Daily,BassLine,Barnes and Sons__0__0__Lomark Daily
2,castillo-elliott,Entity.Organization.FishingCompany,event.fishing,0,tuna shelf,Entity.Location.Region,Junior Shurdlu,2035-02-01 01:00:00,2035-02-01 01:00:00,2035-02-01,Haacklee Herald,BassLine,Castillo-Elliott__0__0__Haacklee Herald
3,city of lomark,Entity.Organization.GovernmentOrg,event.applaud,1,frey inc,Entity.Organization.FishingCompany,Jack Inch,2035-02-01 01:00:00,2035-02-01 01:00:00,2035-02-01,The News Buoy,BassLine,Frey Inc__0__0__The News Buoy
4,barnes and sons,Entity.Organization.FishingCompany,event.transaction,0,ramos-shelton,Entity.Organization.FishingCompany,Kristin Baker,2035-02-01 01:00:00,2035-02-01 01:00:00,2035-02-01,Lomark Daily,BassLine,Barnes and Sons__0__0__Lomark Daily
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14195,sanchez-moreno,Entity.Organization.FishingCompany,event.invest,1,tracking_system,Entity.Commodity,Greta Grass-Hill,2035-07-30 21:00:00,2035-07-30 21:00:00,2035-07-30,Lomark Daily,BassLine,Sanchez-Moreno__0__1__Lomark Daily
14196,sanchez-moreno,Entity.Organization.FishingCompany,event.aid,1,marine_sanctuary,Entity.Organization,Greta Grass-Hill,2035-07-30 21:00:00,2035-07-30 21:00:00,2035-07-30,Lomark Daily,BassLine,Sanchez-Moreno__0__1__Lomark Daily
14197,"taylor, prince and sherman",Entity.Organization.FishingCompany,event.invest,1,efficiency,Entity.Commodity,Worf Peer,2035-07-30 21:00:00,2035-07-30 21:00:00,2035-07-30,Lomark Daily,ShadGPT,"Taylor, Prince and Sherman__0__1__Lomark Daily"
14198,"bell, reynolds and forbes",Entity.Organization.FishingCompany,event.invest,1,efficiency,Entity.Commodity,Olokun Daramola,2035-07-30 22:00:00,2035-07-30 22:00:00,2035-07-30,Lomark Daily,ShadGPT,"Bell, Reynolds and Forbes__0__1__Lomark Daily"


In [7]:
# list( df[ (df["source"]=="Alvarez PLC") & (df["_raw_source"]=="Lomark Daily") ][["source", "type", "target"] ].drop_duplicates().copy().itertuples(index=False, name=None)  )

### Sources

In [8]:
SOURCE_TYPE = df["source_type"].unique().tolist()
SOURCE_TYPE

['Entity.Organization.FishingCompany',
 'Entity.Organization.GovernmentOrg',
 'Entity.Organization.Company',
 'Entity.Organization.LogisticsCompany']

In [9]:
Goverments = df[df["source_type"]== 'Entity.Organization.GovernmentOrg']["source"].unique()

### Targets

In [10]:
SOURCE_TYPE = df["target_type"].unique().tolist()
SOURCE_TYPE

['Entity.Commodity',
 'Entity.Organization.FishingCompany',
 'Entity.Location.Region',
 'Entity.Organization',
 'Entity.Organization.Company',
 'Entity.Organization.NGO',
 'Entity.Organization.LogisticsCompany']

In [11]:
Comodities = df[df["target_type"]== 'Entity.Comodity']["target"].unique()
Locations = df[df["target_type"]== 'Entity.Location.Region']["target"].unique()
Organizations = df[df["target_type"]== 'Entity.Organization']["target"].unique()
NonGoverments = df[df["target_type"]== 'Entity.Organization.NGO']["target"].unique()

In [12]:
a = df[df["target_type"]== 'Entity.Organization.FishingCompany']["target"].unique()
b = df[df["source_type"]== 'Entity.Organization.FishingCompany']["source"].unique()
FishingCompanies = np.unique(np.concatenate((a, b)) ).tolist()

In [13]:
a = df[df["target_type"]== 'Entity.Organization.LogisticsCompany']["target"].unique()
b = df[df["source_type"]== 'Entity.Organization.LogisticsCompany']["source"].unique()
LogisticCompanies = np.unique(np.concatenate((a, b)) ).tolist()

In [14]:
a = df[df["target_type"]== 'Entity.Organization.Company']["target"].unique()
b = df[df["source_type"]== 'Entity.Organization.Company']["source"].unique()
Companies = np.unique(np.concatenate((a, b)) ).tolist()

### Edges

In [15]:
EDGE_TYPES = df["type"].unique().tolist()
EDGE_TYPES

['event.invest',
 'event.certificateissued',
 'event.fishing',
 'event.applaud',
 'event.transaction',
 'event.communication.conference',
 'event.aid',
 'event.fishing.sustainablefishing',
 'event.fishing.overfishing',
 'event.certificateissued.summons',
 'event.criticize',
 'event.convicted']

### LLM-Search

In [16]:
openai.api_key = API_KEY

In [17]:
def extract_information(text, company_talked_about):
    prompt_template = """
                      We want to extract some information from an input text.
                      This text talks about the entity/company {company_talked_about}.
                      For this, we have a list of possible events or discussions: {edge_types},
                      we have a list of companies: {companies}, logistic companies: {logisticcompanies}, and fishing companies: {fishingcompanies}
                      we have a list of goverment entities: {goverments} and non-govermental entities: {nongoverments}, 
                      we have a list of organizations: {organizations}, locations: {locations}, comodities (attributes of companies such as sustainable_nets, tracking_system, efficiency, etc): {comodities}
                      
                      Some samples of discussion topics are:
                      investments or transactions -> Event.Transaction, 
                      news, journals, etc -> Event.Communication.Conference,
                      collaboration partners -> Event.Owns.PartiallyOwns, 
                      Event.Aid -> recommendations, 
                      fishing -> Event.Fishing, 
                      sustainable fishing - > Event.Fishing.SustainableFishing, 
                      ilegal fishing -> Event.Fishing.OverFishing,
                      certified issues or permission to operates -> Event.CertificateIssued, 
                      critics -> Event.Criticize,
                      etc.

                      In this task, we want to identify and extract all information about: 
                      who are they talking about (Source name), 
                      the corresponding event (event type).
                      who is talking (Target name), 
                      what are talking about (Context),

                      Besides, we can obtain more than one event from the same sentence, for example:
                      Example 1:
                      input text: 'Fishing activities primarily take place in designated areas such as Himark and Centralia, where Alvarez PLC operates with a focus on sustainability. By leveraging technological advancements and strategic partnerships, the company continues to thrive in its mission to balance economic viability with environmental preservation.'
                      output: ('City of Himark', 'Event.CertificateIssued', 'Alvarez PLC', 'certified issued and Fishing activities primarily take place'), 
                              ('City of Centralia', 'Event.CertificateIssued', 'Alvarez PLC', 'certified issued and Fishing activities primarily take place')

                      Example 2:
                      input text: 'Marine Sanctuary Aid Boosts Alvarez PLC's Sustainable Fishing Efforts'
                      output: ('Alvarez PLC', 'Event.Aid', 'Marine Sanctuary', 'Aid Boosts')
                      
                      Example 3:
                      input: Alvarez PLC is looking at future of fishing\n\nAlvarez PLC is bolstering its solid reputation as a successful fishing company. It is exploring new partnerships. Cervantes-Kramer is a local fishing company with long-standing permits for fishing in the Wrasse Beds area. The company is committed to sustainable and environmentally friendly fishing. The company has been investing in efficient sustainable nets, and a new tracking system. As of July 2035 it has given $2000 to a local marine sanctuary as aid. It has also signed multiple fishing transactions with Eaton-Osborne and York-Castillo fishing and several logistics companies. It is reported that this company is interested in expanding it's offerings of sustainable fishing.\n\nOther business news:\nColeman, Thompson and Huber is a company to watch. They reported growth of 25% last quarter.
Brown-Stokes will be closed for an upcoming company holiday July 14-17.\nOlson and Sons will transfer company ownership to the family's eldest daughter.
                      output: ('Alvarez PLC', 'Event.Invest', 'Efficiency'), 
                              ('Alvarez PLC', 'Event.Transaction', 'Bell, Reynolds and Forbes'),
                             ('Alvarez PLC', 'Event.Transaction', 'Coleman, Thompson and Huber'),
                             ('Alvarez PLC', 'Event.Transaction', 'Collins, Johnson and Lloyd'),
                             ('Alvarez PLC', 'Event.Invest', 'Safety'),
                             ('Alvarez PLC', 'Event.Transaction', 'Brown-Stokes'),
                             ('Alvarez PLC', 'Event.Transaction', 'Horn and Sons'),
                             ('Alvarez PLC', 'Event.Invest', 'Sustainable_nets'),
                             ('Alvarez PLC', 'Event.Transaction', 'Cuevas PLC'),
                             ('Alvarez PLC', 'Event.Aid', 'Marine_Sanctuary'),
                             ('Alvarez PLC', 'Event.Invest', 'Tracking_System'),
                             ('Alvarez PLC', 'Event.Transaction', 'Allen-Weiss'),
                             ('Alvarez PLC', 'Event.Transaction', 'Ross-Curtis'),
                             ('Alvarez PLC', 'Event.Transaction', 'Olson and Sons'),
                             ('Alvarez PLC', 'Event.Transaction', 'Bates-Anderson'),
                             ('Alvarez PLC', 'Event.Transaction', 'Lee-Smith'),
                             ('Alvarez PLC', 'Event.Transaction', 'Lopez-Delgado'),
                             ('Alvarez PLC', 'Event.Transaction', 'Klein LLC'),
                             ('Alvarez PLC', 'Event.Transaction', 'Ward-Dunn'),
                             ('Alvarez PLC', 'Event.Transaction', 'Glover, Moran and Johnson')

                      From this, could you identify what kind of events are being discussed in the following text: {text}
                      please ensure to return the extracted information result in the the following format:
                          (Source name, Event type, Target name, Context)
                      Extracted information:
                      """
    prompt_request = prompt_template.format(text=text, 
                                            edge_types=EDGE_TYPES, 
                                            company_talked_about=company_talked_about, 
                                            companies=Companies, 
                                            logisticcompanies=LogisticCompanies, 
                                            fishingcompanies=FishingCompanies,
                                            goverments=Goverments,
                                            nongoverments=NonGoverments,
                                            comodities=Comodities,
                                            locations=Locations,
                                            organizations=Organizations
                                           )
    #print("Text length", len(prompt_request))
    
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",  # Use a supported model
        messages=[
            {"role": "system", "content": "You are an information extraction assistant."},
            {"role": "user", "content": f"Below, there is a raw text which where each line is a news or description about the company {company_talked_about}"},
            {"role": "user", "content": prompt_request},
        ],
        max_tokens=500,
        temperature=0.5,
    )
    
    return response, response['choices'][0]['message']['content'].strip()

In [18]:
texts = np.sort(glob.glob(f"{ARTICLES_PATH}*.txt"))
len(texts)

338

In [19]:
data ={"result": [], "_articleid": []}

for index, text_path in tqdm(enumerate(texts)):

    if "Police" in text_path.split("/")[-1]:
        continue
    
    with open(f'{text_path}', 'r') as file:
        # Read the entire contents of the file
        contents = file.read()

    article_id = text_path.split("/")[-1].replace(".txt", "")
    company_name = text_path.split("/")[-1].split("__")[0]
    reviewer_name = text_path.split("/")[-1].split("__")[-1]
    contents = contents.replace("'", "")
    _, extracted_info = extract_information(contents, company_name)

    result = [txt for txt in extracted_info.replace("Extracted information:", "").split("\n") if len(txt)>0]

    data["result"].append(result)
    data["_articleid"].append(article_id)
    # # Loop through each item in the list
    # print("Attemp", index, result)
    # for item in result:
    #     item = item.replace("[", "{").replace("]", "}")
    #     print(item, "\n\n\n")
    #     # Split the string at the first occurrence of the period and space to separate the ID from the JSON string
    #     try:
    #         id, json_str = item.split('. ', 1)
    #     except:
    #         json_str = item
        
    #     # Parse the JSON string into a dictionary
    #     event_data = json.loads(json_str)
        
    #     # Add the parsed dictionary to the result dictionary with the ID as the key
    #     data.append( event_data )
    

338it [21:00,  3.73s/it]


In [20]:
final_data={"source":[], "type":[], "target":[], "context":[], "_articleid": []}
error_count = 0

for items in tqdm(zip(data["_articleid"], data["result"])):
    article_id = items[0]
    links = items[1]
    for item in links:
        if ")" not in item:
            item = item+"')"
        pattern = re.compile(r"\((.*?)\)")
        pattern_number = r"\d+\. "

        cleaned_text = re.sub(pattern_number, "", item)
        
        cleaned_text = cleaned_text.replace("- ", "")\
                                   .replace("company's", "companys")\
                                   .replace("Fisherman's", "Fishermans")\
                                   .replace("Inc’s", "Incs")
        cleaned_text = cleaned_text.replace("((", "(").replace("),)", ")")
        #item = "("+pattern.search(re.sub(r"[\\']", '', str(item).replace("\\", "")) ).group(1) + ")"
        try:
            
            if ". (" in cleaned_text:
                _text = cleaned_text.split(". ")
                print(cleaned_text, _text)
            
            extract_values = ast.literal_eval(str(cleaned_text))
            if len(extract_values)==1:
                extract_values=extract_values[0]

            if len(extract_values)==2:
                continue
        
            context = extract_values[3] if len(extract_values)==4 else ""
            final_data["source"].append(extract_values[0])
            final_data["type"].append(extract_values[1])
            final_data["target"].append(extract_values[2])
            final_data["context"].append(context)
            final_data["_articleid"].append(article_id)
            
        except Exception as e:
            error_count+=1
            print(f"Error: {e}\nraw text: {item}\nclean:{cleaned_text}\n\n\n")
            
            if len(extract_values)<3:
                continue
            context = extract_values[3] if len(extract_values)==4 else ""
            final_data["source"].append(extract_values[0])
            final_data["type"].append(extract_values[1])
            final_data["target"].append(extract_values[2])
            final_data["context"].append(context)
            final_data["_articleid"].append(article_id)
            print(extract_values[0])
            
            
print(f"Total errors: {error_count}")

328it [00:00, 12822.31it/s]

Error: invalid syntax (<unknown>, line 1)
raw text: The extracted information from the text "Anderson, Brown and Green Champions Sustainable Fishing Practices" is as follows:')
clean:The extracted information from the text "Anderson, Brown and Green Champions Sustainable Fishing Practices" is as follows:')



Anderson, Brown and Green
Error: invalid syntax (<unknown>, line 1)
raw text: 1. (Anderson, Brown and Green, Event.Aid, Marine Sanctuary, Investment in a tracking system and participation in sustainable fishing activities in Wrasse Beds)
clean:(Anderson, Brown and Green, Event.Aid, Marine Sanctuary, Investment in a tracking system and participation in sustainable fishing activities in Wrasse Beds)



Anderson, Brown and Green
Error: malformed node or string: <ast.Name object at 0x7fb895f57bb0>
raw text: 2. (Anderson, Brown and Green, Event.Communication.Conference, Thomas-Weaver, Participation in conferences)
clean:(Anderson, Brown and Green, Event.Communication.Conference, Thomas

In [21]:
for k,v in final_data.items():
    print(k, len(final_data[k]) )

source 2820
type 2820
target 2820
context 2820
_articleid 2820


In [22]:
ourGPT = pd.DataFrame(data=final_data)
ourGPT

,source,type,target,context,_articleid
0,Alvarez PLC,Event.Aid,Marine Sanctuary,Aid to preserve vital ecosystems,Alvarez PLC__0__0__Haacklee Herald
1,Alvarez PLC,Event.Invest,Efficiency,Strategic investments in efficiency and safety...,Alvarez PLC__0__0__Haacklee Herald
2,Alvarez PLC,Event.Invest,Safety,Strategic investments in efficiency and safety...,Alvarez PLC__0__0__Haacklee Herald
3,Alvarez PLC,Event.Invest,Sustainable_nets,Development of sustainable nets,Alvarez PLC__0__0__Haacklee Herald
4,Alvarez PLC,Event.Invest,Tracking_System,Development of tracking systems,Alvarez PLC__0__0__Haacklee Herald
...,...,...,...,...,...
2815,York-Castillo,Event.Transaction,Cervantes-Kramer,signed multiple fishing transactions with Cerv...,York-Castillo__0__0__The News Buoy
2816,York-Castillo,Event.Fishing,City Of Lomark,applauded for its commitment to sustainable an...,York-Castillo__0__0__The News Buoy
2817,York-Castillo,Event.Fishing,City Of South Paackland,applauded for its commitment to sustainable an...,York-Castillo__0__0__The News Buoy
2818,York-Castillo,Event.CertificateIssued,City Of Lomark,"approved allowed to do fishing in Wrasse Beds,...",York-Castillo__0__0__The News Buoy


In [23]:
ourGPT.to_csv(f"{CSV_PATH}graph.csv", sep="\t", index=False)

### Finding missing values

In [24]:
ourGPT = pd.read_csv(f"{CSV_PATH}graph.csv", sep="\t", low_memory=False)
ourGPT = ourGPT[~ourGPT["target"].isna()]
ourGPT["source"] = ourGPT["source"].apply(lambda x: x.lower())
ourGPT["type"] = ourGPT["type"].apply(lambda x: x.lower())
ourGPT["target"] = ourGPT["target"].apply(lambda x: x.lower())
ourGPT

,source,type,target,context,_articleid
0,alvarez plc,event.aid,marine sanctuary,Aid to preserve vital ecosystems,Alvarez PLC__0__0__Haacklee Herald
1,alvarez plc,event.invest,efficiency,Strategic investments in efficiency and safety...,Alvarez PLC__0__0__Haacklee Herald
2,alvarez plc,event.invest,safety,Strategic investments in efficiency and safety...,Alvarez PLC__0__0__Haacklee Herald
3,alvarez plc,event.invest,sustainable_nets,Development of sustainable nets,Alvarez PLC__0__0__Haacklee Herald
4,alvarez plc,event.invest,tracking_system,Development of tracking systems,Alvarez PLC__0__0__Haacklee Herald
...,...,...,...,...,...
2815,york-castillo,event.transaction,cervantes-kramer,signed multiple fishing transactions with Cerv...,York-Castillo__0__0__The News Buoy
2816,york-castillo,event.fishing,city of lomark,applauded for its commitment to sustainable an...,York-Castillo__0__0__The News Buoy
2817,york-castillo,event.fishing,city of south paackland,applauded for its commitment to sustainable an...,York-Castillo__0__0__The News Buoy
2818,york-castillo,event.certificateissued,city of lomark,"approved allowed to do fishing in Wrasse Beds,...",York-Castillo__0__0__The News Buoy


Without Context:

```
[txt for txt in extracted_info.replace("Extracted information:", "").split("\n") if len(txt)>0]
```

response:

['1. (Alvarez PLC, Marine Sanctuary, Event.Aid)',  
 '2. (Alvarez PLC, Alvarez PLC, Event.Fishing.SustainableFishing)',  
 '3. (Alvarez PLC, Clements, Allen and Sullivan, Event.Communication.Conference)',  
 '4. (Alvarez PLC, Franco-Stuart, Event.Communication.Conference)']

With Context:

```
[txt for txt in extracted_info.replace("Extracted information:", "").split("\n") if len(txt)>0]
```

response:

['1. (Alvarez PLC, Marine Sanctuary, Providing aid to preserve vital ecosystems, Event.Aid)',  
 '2. (Alvarez PLC, Clements, Allen and Sullivan, Participation in conferences for knowledge exchange, Event.Communication.Conference)',  
 '3. (Alvarez PLC, Franco-Stuart, Participation in conferences for knowledge exchange, Event.Communication.Conference)',  
 '4. (Alvarez PLC, Strategic investments in efficiency and safety commodities, Underscoring commitment to responsible fishing practices, Event.Invest)',  
 '5. (Alvarez PLC, Development of sustainable nets and tracking systems, Enhancing environmental stewardship, Event.Fishing.SustainableFishing)',  
 '6. (Alvarez PLC, Collaborations with Barnett Ltd and Cain, Simpson, and Hernandez, Expanding influence in the market, Event.Transaction)',  
 '7. (Alvarez PLC, Collaborations with Maldonado, Sanchez, and Jones, Expanding influence in the market, Event.Transaction)',  
 '8. (Alvarez PLC, Collaborations with Johnson-Johnson, Expanding influence in the market, Event.Transaction)']